## Import and preprocessing

Import

In [350]:
%load_ext autoreload
%autoreload 2

from importlib import reload
import my_sklearn
reload(my_sklearn)

from my_sklearn import my_LinearRegression, my_Lasso, my_Ridge, my_ElasticNet, my_MinMaxScaler, my_StandardScaler

import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from collections import Counter
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Read train and test json files, downloaded from [kaggle.com](https://www.kaggle.com/competitions/two-sigma-connect-rental-listing-inquiries/data?select=train.json.zip)

In [351]:
df = pd.read_json("data/train.json").reset_index()

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49352 entries, 0 to 49351
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            49352 non-null  int64  
 1   bathrooms        49352 non-null  float64
 2   bedrooms         49352 non-null  int64  
 3   building_id      49352 non-null  str    
 4   created          49352 non-null  str    
 5   description      49352 non-null  str    
 6   display_address  49352 non-null  str    
 7   features         49352 non-null  object 
 8   latitude         49352 non-null  float64
 9   listing_id       49352 non-null  int64  
 10  longitude        49352 non-null  float64
 11  manager_id       49352 non-null  str    
 12  photos           49352 non-null  object 
 13  price            49352 non-null  int64  
 14  street_address   49352 non-null  str    
 15  interest_level   49352 non-null  str    
dtypes: float64(3), int64(4), object(2), str(7)
memory usage: 6.0+ MB


Generating additional features for better model quality

In [352]:
df['Features'] = df['features'].apply(
    lambda x: str(x)
    .replace('[', '')
    .replace(']', '')
    .replace('"', '')
    .replace("'", '')
    .replace(' ', '')
)

In [353]:
features = (
    df['Features']
    .str.split(',')
    .explode()
    .loc[lambda s: s != '']
    .tolist()
)
unique = set(features)
len(unique)

1546

In [354]:
top_feat = Counter(features).most_common(20)
top_feat

[('Elevator', 25915),
 ('CatsAllowed', 23540),
 ('HardwoodFloors', 23527),
 ('DogsAllowed', 22035),
 ('Doorman', 20898),
 ('Dishwasher', 20426),
 ('NoFee', 18062),
 ('LaundryinBuilding', 16344),
 ('FitnessCenter', 13252),
 ('Pre-War', 9148),
 ('LaundryinUnit', 8738),
 ('RoofDeck', 6542),
 ('OutdoorSpace', 5268),
 ('DiningRoom', 5136),
 ('HighSpeedInternet', 4299),
 ('Balcony', 2992),
 ('SwimmingPool', 2730),
 ('LaundryInBuilding', 2593),
 ('NewConstruction', 2559),
 ('Terrace', 2283)]

In [355]:
feature_list = [ftr[0] for ftr in top_feat]

X = pd.DataFrame()

for feature in feature_list:
    X[feature] = df['Features'].str.contains(feature, regex=False).astype('int8')

X['bathrooms'] = df['bathrooms']
X['bedrooms'] = df['bedrooms']

feature_list += ['bathrooms', 'bedrooms']

y = df['price']

## Models implementation

### Linear Regression

In [356]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=21,
    test_size=0.2
)

low = y_train.quantile(0.01)
high = y_train.quantile(0.99)
mask = (y_train >= low) & (y_train <= high)

X_train = X_train[mask]
y_train = y_train[mask]

result_R2 = pd.DataFrame(columns=['model', 'train', 'test'])
result_MAE = pd.DataFrame(columns=['model', 'train', 'test'])
result_RMSE = pd.DataFrame(columns=['model', 'train', 'test'])

In [357]:
def my_R2(y, y_pred):

    y = np.asarray(y)
    y_pred = np.asarray(y_pred)

    if y.shape != y_pred.shape:
        raise ValueError('must be the same shape')

    y_mean = np.mean(y)
    tss = np.sum((y - y_mean)**2)
    
    if np.isclose(tss, 0):
        raise ValueError("TSS must be not 0")
    
    rss = np.sum((y - y_pred)**2)

    return 1 - rss / tss

def evaluate_model(model, name, result_MAE, result_RMSE, result_R2, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    MAE_test = mean_absolute_error(y_test, y_pred_test)
    MAE_train = mean_absolute_error(y_train, y_pred_train)

    RMSE_test = root_mean_squared_error(y_test, y_pred_test)
    RMSE_train = root_mean_squared_error(y_train, y_pred_train)

    R2_train = my_R2(y_train, y_pred_train)
    R2_test = my_R2(y_test, y_pred_test)

    result_MAE.loc[len(result_MAE)] = [name, MAE_train, MAE_test]
    result_RMSE.loc[len(result_RMSE)] = [name, RMSE_train, RMSE_test]
    result_R2.loc[len(result_R2)] = [name, R2_train, R2_test]

def results(result_MAE, result_RMSE, result_R2):
    print("result_MAE")
    display(result_MAE.sort_values(by='train'))

    print("result_RMSE")
    display(result_RMSE.sort_values(by='train'))

    print("result_R2")
    display(result_R2.sort_values(by='train', ascending=False))


#### sklearn linear regression

In [358]:
model = LinearRegression()
name = 'LinearRegression'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

#### my implementation

In [359]:
model = my_LinearRegression(random_state=21)
name = 'my_LinearRegression'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [360]:
model = my_LinearRegression(random_state=21, solver='gd')
name = 'my_LinearRegression_gd'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [361]:
model = my_LinearRegression(random_state=21, solver='analytical')
name = 'my_LinearRegression_analytical'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [362]:
results(result_MAE, result_RMSE, result_R2)

result_MAE


,model,train,test
3,my_LinearRegression_analytical,711.042070,816.715299
0,LinearRegression,711.042070,816.715299
1,my_LinearRegression,718.754908,823.630096
2,my_LinearRegression_gd,736.901174,842.275038


result_RMSE


,model,train,test
0,LinearRegression,1034.777295,2148.039048
3,my_LinearRegression_analytical,1034.777295,2148.039048
1,my_LinearRegression,1050.531278,2154.387514
2,my_LinearRegression_gd,1075.505816,2203.656589


result_R2


,model,train,test
0,LinearRegression,0.581343,0.353769
3,my_LinearRegression_analytical,0.581343,0.353769
1,my_LinearRegression,0.568498,0.349943
2,my_LinearRegression_gd,0.547738,0.319871


### Ridge, Lasso, ElasticNet

In [363]:
model = my_Ridge(random_state=21)
name = 'my_Ridge'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [364]:
model = my_Ridge(random_state=21, solver='analytical')
name = 'my_Ridge_analytical'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [365]:
model = my_Lasso(random_state=21)
name = 'my_Lasso'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [366]:
model = my_ElasticNet(random_state=21)
name = 'my_ElasticNet'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [367]:
model = Ridge()
name = 'Ridge'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [368]:
model = Lasso()
name = 'Lasso'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [369]:
model = ElasticNet()
name = 'ElasticNet'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [370]:
results(result_MAE, result_RMSE, result_R2)

result_MAE


,model,train,test
9,Lasso,710.602794,815.691056
8,Ridge,711.037998,816.711853
5,my_Ridge_analytical,711.037998,816.711853
3,my_LinearRegression_analytical,711.042070,816.715299
0,LinearRegression,711.042070,816.715299
6,my_Lasso,718.297203,823.087532
1,my_LinearRegression,718.754908,823.630096
2,my_LinearRegression_gd,736.901174,842.275038
7,my_ElasticNet,781.292701,897.949723
10,ElasticNet,807.022377,925.305999


result_RMSE


,model,train,test
0,LinearRegression,1034.777295,2148.039048
3,my_LinearRegression_analytical,1034.777295,2148.039048
5,my_Ridge_analytical,1034.777303,2148.058864
8,Ridge,1034.777303,2148.058864
9,Lasso,1034.968775,2148.728854
6,my_Lasso,1050.335659,2154.599574
1,my_LinearRegression,1050.531278,2154.387514
2,my_LinearRegression_gd,1075.505816,2203.656589
7,my_ElasticNet,1161.687631,2305.821990
10,ElasticNet,1190.383118,2341.473096


result_R2


,model,train,test
0,LinearRegression,0.581343,0.353769
3,my_LinearRegression_analytical,0.581343,0.353769
5,my_Ridge_analytical,0.581343,0.353757
8,Ridge,0.581343,0.353757
9,Lasso,0.581188,0.353354
6,my_Lasso,0.568659,0.349815
1,my_LinearRegression,0.568498,0.349943
2,my_LinearRegression_gd,0.547738,0.319871
7,my_ElasticNet,0.472353,0.255345
10,ElasticNet,0.445963,0.232140


### Feature normalization

Compare the feature normalization with your own method and with sklearn.

MinMaxScaler

In [371]:
my_scaler = my_MinMaxScaler()

my_scaler.fit(X_train)

my_X_train_scaled_minmax = my_scaler.transform(X_train)
my_X_test_scaled_minmax = my_scaler.transform(X_test)

scaler = MinMaxScaler()

scaler.fit(X_train)

X_train_scaled_minmax = my_scaler.transform(X_train)
X_test_scaled_minmax = my_scaler.transform(X_test)

Comparison Using `np.allclose`

`np.allclose()` is a NumPy function that checks whether two arrays are element-wise approximately equal within a specified tolerance.

Parameters:

- `a`, `b` — array-like inputs to compare.
- `rtol` — relative tolerance (default: $10^{-5}$).
- `atol` — absolute tolerance (default: $10^{-8}$).

Two values are considered equal if:

$$
|a - b| \leq atol + rtol \cdot |b|
$$

where:

- $a$ and $b$ are the elements being compared.
- $atol$ (Absolute Tolerance) is the minimum allowed tolerance. It is especially important for values close to zero.
- $rtol$ (Relative Tolerance) scales the tolerance according to the magnitude of the reference value $b$. It is useful when comparing larger numbers, where a fixed absolute tolerance may be too strict.

In [372]:
print(np.allclose(my_X_train_scaled_minmax, X_train_scaled_minmax))
print(np.allclose(my_X_test_scaled_minmax, X_test_scaled_minmax))

True
True


StandardScaler

In [373]:
my_scaler = my_StandardScaler()

my_scaler.fit(X_train)

my_X_train_scaled = my_scaler.transform(X_train)
my_X_test_scaled = my_scaler.transform(X_test)

scaler = StandardScaler()

scaler.fit(X_train)

X_train_scaled = my_scaler.transform(X_train)
X_test_scaled = my_scaler.transform(X_test)

In [374]:
print(np.allclose(my_X_train_scaled, X_train_scaled))
print(np.allclose(my_X_test_scaled, X_test_scaled))

True
True


### Fit custom and sklearn models with normalized data

MinMaxScaler

In [375]:
models = {
    'my_LinearRegression_minmax' : my_LinearRegression(random_state=21),
    'my_Ridge_minmax' : my_Ridge(random_state=21),
    'my_Lasso_minmax' : my_Lasso(random_state=21),
    'my_ElasticNet_minmax' : my_ElasticNet(random_state=21),
    'LinearRegression_minmax' : LinearRegression(),
    'Ridge_minmax' : Ridge(),
    'Lasso_minmax' : Lasso(),
    'ElasticNet_minmax' : ElasticNet()
}

for name, model in models.items():
    evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    my_X_train_scaled_minmax, my_X_test_scaled_minmax, y_train, y_test
)

In [376]:
models = {
    'my_LinearRegression_scaled' : my_LinearRegression(random_state=21),
    'my_Ridge_scaled' : my_Ridge(random_state=21),
    'my_Lasso_scaled' : my_Lasso(random_state=21),
    'my_ElasticNet_scaled' : my_ElasticNet(random_state=21),
    'LinearRegression_scaled' : LinearRegression(),
    'Ridge_scaled' : Ridge(),
    'Lasso_scaled' : Lasso(),
    'ElasticNet_scaled' : ElasticNet()
}

for name, model in models.items():
    evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    my_X_train_scaled, my_X_test_scaled, y_train, y_test
)

In [377]:
results(result_MAE, result_RMSE, result_R2)

result_MAE


,model,train,test
9,Lasso,710.602794,815.691056
13,my_Lasso_minmax,710.793300,816.392232
25,Lasso_scaled,710.839901,816.320611
17,Lasso_minmax,710.858936,816.565728
8,Ridge,711.037998,816.711853
5,my_Ridge_analytical,711.037998,816.711853
24,Ridge_scaled,711.041359,816.715010
3,my_LinearRegression_analytical,711.042070,816.715299
15,LinearRegression_minmax,711.042070,816.715299
23,LinearRegression_scaled,711.042070,816.715299


result_RMSE


,model,train,test
0,LinearRegression,1034.777295,2148.039048
3,my_LinearRegression_analytical,1034.777295,2148.039048
15,LinearRegression_minmax,1034.777295,2148.039048
23,LinearRegression_scaled,1034.777295,2148.039048
24,Ridge_scaled,1034.777295,2148.042784
5,my_Ridge_analytical,1034.777303,2148.058864
8,Ridge,1034.777303,2148.058864
25,Lasso_scaled,1034.806389,2148.294617
16,Ridge_minmax,1034.828217,2149.824721
9,Lasso,1034.968775,2148.728854


result_R2


,model,train,test
0,LinearRegression,0.581343,0.353769
3,my_LinearRegression_analytical,0.581343,0.353769
15,LinearRegression_minmax,0.581343,0.353769
23,LinearRegression_scaled,0.581343,0.353769
24,Ridge_scaled,0.581343,0.353766
8,Ridge,0.581343,0.353757
5,my_Ridge_analytical,0.581343,0.353757
25,Lasso_scaled,0.581319,0.353615
16,Ridge_minmax,0.581301,0.352694
9,Lasso,0.581188,0.353354


### Overfit models

In [378]:
poly = PolynomialFeatures(degree=30)

X_train_for_poly = X_train[['bathrooms', 'bedrooms']]
X_test_for_poly = X_test[['bathrooms', 'bedrooms']]

poly.fit(X_train_for_poly)

X_train_poly = poly.transform(X_train_for_poly)
X_test_poly = poly.transform(X_test_for_poly)

scaler = my_MinMaxScaler()
scaler.fit(X_train_poly)

X_train_poly = scaler.transform(X_train_poly)
X_test_poly = scaler.transform(X_test_poly)

result_R2_poly = pd.DataFrame(columns=['model', 'train', 'test'])
result_MAE_poly = pd.DataFrame(columns=['model', 'train', 'test'])
result_RMSE_poly = pd.DataFrame(columns=['model', 'train', 'test'])

In [379]:
models = {
    'my_LinearRegression_poly' : my_LinearRegression(random_state=21, solver='analytical'),
    'my_Ridge_poly2' : my_Ridge(random_state=21, n_epochs=800, solver='analytical'),
    'my_Lasso_poly2' : my_Lasso(random_state=21, n_epochs=800),
    'my_ElasticNet_poly' : my_ElasticNet(random_state=21,  n_epochs=800),
}

for name, model in models.items():
    evaluate_model(
    model, name,
    result_MAE_poly, result_RMSE_poly, result_R2_poly,
    X_train_poly, X_test_poly, y_train, y_test
)

In [380]:
results(result_MAE_poly, result_RMSE_poly, result_R2_poly)

result_MAE


,model,train,test
0,my_LinearRegression_poly,757.220379,233641.875119
1,my_Ridge_poly2,775.355073,869.383841
2,my_Lasso_poly2,776.624602,884.915773
3,my_ElasticNet_poly,1101.894979,1237.894425


result_RMSE


,model,train,test
0,my_LinearRegression_poly,1079.070813,2.296788e+07
1,my_Ridge_poly2,1107.753539,2.012018e+03
2,my_Lasso_poly2,1122.550783,2.382778e+03
3,my_ElasticNet_poly,1558.368704,2.638468e+03


result_R2


,model,train,test
0,my_LinearRegression_poly,0.544734,-7.388317e+07
1,my_Ridge_poly2,0.520210,4.330204e-01
2,my_Lasso_poly2,0.507306,2.048098e-01
3,my_ElasticNet_poly,0.050477,2.499349e-02


#### Additionally try different alpha parameters of regularization in algorithms, choose the best one and analyze results.

In [381]:
alpha = [0.001, 0.01, 0.1, 1, 10, 100]



for a in alpha:
    models = {
    f'my_Ridge_poly_{a}' : my_Ridge(random_state=21, alpha=a, solver='analytical'),
    f'my_Lasso_poly_{a}' : my_Lasso(random_state=21, alpha=a),
    f'my_ElasticNet_poly_{a}' : my_ElasticNet(random_state=21, alpha=a),
    }

    for name, model in models.items():
        evaluate_model(
        model, name,
        result_MAE_poly, result_RMSE_poly, result_R2_poly,
        X_train_poly, X_test_poly, y_train, y_test
    )

In [382]:
results(result_MAE_poly, result_RMSE_poly, result_R2_poly)

result_MAE


,model,train,test
0,my_LinearRegression_poly,757.220379,233641.875119
4,my_Ridge_poly_0.001,761.425033,945.799130
7,my_Ridge_poly_0.01,762.357761,858.247393
10,my_Ridge_poly_0.1,766.253964,857.556394
13,my_Ridge_poly_1,775.355073,869.383841
1,my_Ridge_poly2,775.355073,869.383841
2,my_Lasso_poly2,776.624602,884.915773
14,my_Lasso_poly_1,776.624602,884.915773
16,my_Ridge_poly_10,782.252219,884.471702
17,my_Lasso_poly_10,789.065544,892.541125


result_RMSE


,model,train,test
0,my_LinearRegression_poly,1079.070813,2.296788e+07
4,my_Ridge_poly_0.001,1087.142557,9.360788e+03
7,my_Ridge_poly_0.01,1089.557778,2.026212e+03
10,my_Ridge_poly_0.1,1095.803469,2.001357e+03
13,my_Ridge_poly_1,1107.753539,2.012018e+03
1,my_Ridge_poly2,1107.753539,2.012018e+03
16,my_Ridge_poly_10,1120.099146,2.221532e+03
2,my_Lasso_poly2,1122.550783,2.382778e+03
14,my_Lasso_poly_1,1122.550783,2.382778e+03
17,my_Lasso_poly_10,1132.281224,2.194156e+03


result_R2


,model,train,test
0,my_LinearRegression_poly,0.544734,-7.388317e+07
4,my_Ridge_poly_0.001,0.537898,-1.127237e+01
7,my_Ridge_poly_0.01,0.535842,4.249923e-01
10,my_Ridge_poly_0.1,0.530506,4.390130e-01
1,my_Ridge_poly2,0.520210,4.330204e-01
13,my_Ridge_poly_1,0.520210,4.330204e-01
16,my_Ridge_poly_10,0.509456,3.087918e-01
14,my_Lasso_poly_1,0.507306,2.048098e-01
2,my_Lasso_poly2,0.507306,2.048098e-01
17,my_Lasso_poly_10,0.498728,3.257226e-01


- Best model:

    - The best model is Ridge regression with α ≈ 0.1.

- Why:

    - It provides the best trade-off between bias and variance
    - It significantly reduces overfitting caused by polynomial features
    - It is stable across different runs and hyperparameters
    - It achieves the lowest MAE on test data among stable models

- Effect of alpha:

    - Small α → weak regularization → overfitting
    - Medium α (0.1–1) → best generalization
    - Large α (100–500) → underfitting

### Native models

In [383]:
y_pred_mean_train = np.full(len(y_train), y_train.mean())
y_pred_mean_test  = np.full(len(y_test), y_train.mean())
y_pred_median_train = np.full(len(y_train), y_train.median())
y_pred_median_test = np.full(len(y_test), y_train.median())

result_MAE.loc[len(result_MAE)] = [
    "NAIVE_MEAN",
    mean_absolute_error(y_train, y_pred_mean_train),
    mean_absolute_error(y_test, y_pred_mean_test)
]

result_RMSE.loc[len(result_RMSE)] = [
    "NAIVE_MEAN",
    root_mean_squared_error(y_train, y_pred_mean_train),
    root_mean_squared_error(y_test, y_pred_mean_test)
]

result_R2.loc[len(result_R2)] = [
    "NAIVE_MEAN",
    my_R2(y_train, y_pred_mean_train),
    my_R2(y_test, y_pred_mean_test)
]

result_MAE.loc[len(result_MAE)] = [
    "NAIVE_MEDIAN",
    mean_absolute_error(y_train, y_pred_median_train),
    mean_absolute_error(y_test, y_pred_median_test)
]

result_RMSE.loc[len(result_RMSE)] = [
    "NAIVE_MEDIAN",
    root_mean_squared_error(y_train, y_pred_median_train),
    root_mean_squared_error(y_test, y_pred_median_test)
]

result_R2.loc[len(result_R2)] = [
    "NAIVE_MEDIAN",
    my_R2(y_train, y_pred_median_train),
    my_R2(y_test, y_pred_median_test)
]

### Compare results

In [384]:
results(result_MAE, result_RMSE, result_R2)

result_MAE


,model,train,test
9,Lasso,710.602794,815.691056
13,my_Lasso_minmax,710.793300,816.392232
25,Lasso_scaled,710.839901,816.320611
17,Lasso_minmax,710.858936,816.565728
8,Ridge,711.037998,816.711853
5,my_Ridge_analytical,711.037998,816.711853
24,Ridge_scaled,711.041359,816.715010
3,my_LinearRegression_analytical,711.042070,816.715299
15,LinearRegression_minmax,711.042070,816.715299
23,LinearRegression_scaled,711.042070,816.715299


result_RMSE


,model,train,test
0,LinearRegression,1034.777295,2148.039048
3,my_LinearRegression_analytical,1034.777295,2148.039048
15,LinearRegression_minmax,1034.777295,2148.039048
23,LinearRegression_scaled,1034.777295,2148.039048
24,Ridge_scaled,1034.777295,2148.042784
5,my_Ridge_analytical,1034.777303,2148.058864
8,Ridge,1034.777303,2148.058864
25,Lasso_scaled,1034.806389,2148.294617
16,Ridge_minmax,1034.828217,2149.824721
9,Lasso,1034.968775,2148.728854


result_R2


,model,train,test
0,LinearRegression,0.581343,0.353769
3,my_LinearRegression_analytical,0.581343,0.353769
15,LinearRegression_minmax,0.581343,0.353769
23,LinearRegression_scaled,0.581343,0.353769
24,Ridge_scaled,0.581343,0.353766
8,Ridge,0.581343,0.353757
5,my_Ridge_analytical,0.581343,0.353757
25,Lasso_scaled,0.581319,0.353615
16,Ridge_minmax,0.581301,0.352694
9,Lasso,0.581188,0.353354


Final conclusion

1. Comparison with sklearn implementations

The implemented models (my_LinearRegression, my_Ridge, my_Lasso, my_ElasticNet) show results that are practically identical to sklearn counterparts.

Differences in MAE, RMSE, and R² are negligible (only in very small decimal ranges)
This confirms that:
Gradient descent optimization is implemented correctly
Analytical solutions match the closed-form solutions used in sklearn
Regularization terms behave as expected

Overall conclusion: the custom implementations are consistent with sklearn models

2. Best model

The best performance is achieved by:

Linear Regression
Ridge Regression
Lasso Regression

All three models show very close results on the test set:

MAE ≈ 815–817
RMSE ≈ 2148
R² ≈ 0.353–0.354

The differences between them are minimal, so no single model significantly outperforms the others.

3. Most stable model

The most stable model is Ridge Regression.

Reasoning:

Consistent performance across different preprocessing methods (standard scaling, min-max scaling, raw features)
Small gap between training and test metrics
Reduced sensitivity to multicollinearity introduced by polynomial features
More robust behavior compared to Linear Regression and Lasso

### Addition task

#### Using a monotone function to "improve" the distribution
log1p is log(1 + y). It's safe for zeros

In [385]:
y_train_log = np.log1p(y_train)

In [386]:
model = my_LinearRegression(random_state=21, solver='analytical')
model.fit(X_train, y_train_log)

y_pred_train_log = model.predict(X_train)
y_pred_train = np.expm1(y_pred_train_log)

y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)

print(f"MAE: train - {mean_absolute_error(y_train, y_pred_train)} | test - {mean_absolute_error(y_test, y_pred)}")
print(f"RMSE: train - {root_mean_squared_error(y_train, y_pred_train)} | test - {root_mean_squared_error(y_test, y_pred)}")
print(f"R2: train - {my_R2(y_train, y_pred_train)} | test - {my_R2(y_test, y_pred)}")

MAE: train - 697.3923902916548 | test - 789.4410714635899
RMSE: train - 1064.181153500657 | test - 2030.5399887595506
R2: train - 0.5572117450903993 | test - 0.4225335068878582


#### Outliers

We remove outliers only from the training data, while keeping the test set unchanged.

- The test set must simulate the real world

    - The test set represents new, unseen data.

    - If we modify it (e.g., remove outliers), then:
    it no longer reflects real data distribution
    evaluation becomes unrealistic
    results become overly optimistic
    - Prevent data leakage

- If we clean the test set, we are indirectly using information about it during preprocessing decisions and this leads to:

    - data leakage
    - inflated performance metrics
    - misleading conclusions about model quality

In [387]:
Q1 = y_train.quantile(0.25)
Q3 = y_train.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

mask = (y_train >= lower) & (y_train <= upper)

X_train_clean = X_train[mask]
y_train_clean = y_train[mask]

In [388]:
model = my_LinearRegression(random_state=21, solver='analytical')
model.fit(X_train_clean, y_train_clean)

y_pred_train_clean = model.predict(X_train_clean)
y_pred_test_clean = model.predict(X_test)

print(f"MAE: train - {mean_absolute_error(y_train_clean, y_pred_train_clean)} | test - {mean_absolute_error(y_test, y_pred_test_clean)}")
print(f"RMSE: train - {root_mean_squared_error(y_train_clean, y_pred_train_clean)} | test - {root_mean_squared_error(y_test, y_pred_test_clean)}")
print(f"R2: train - {my_R2(y_train_clean, y_pred_train_clean)} | test - {my_R2(y_test, y_pred_test_clean)}")

MAE: train - 579.7056671022909 | test - 840.8969948258366
RMSE: train - 758.1645920468291 | test - 2277.0373367774187
R2: train - 0.5106282820314362 | test - 0.2738204743683951


#### Linear regression algorithm with batch and mini-batch training or analytical solution

In [389]:
model = my_LinearRegression(random_state=21, solver='mbgd')
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f"MAE: train - {mean_absolute_error(y_train, y_pred_train)} | test - {mean_absolute_error(y_test, y_pred_test)}")
print(f"RMSE: train - {root_mean_squared_error(y_train, y_pred_train)} | test - {root_mean_squared_error(y_test, y_pred_test)}")
print(f"R2: train - {my_R2(y_train, y_pred_train)} | test - {my_R2(y_test, y_pred_test)}")

MAE: train - 712.9027247786019 | test - 817.533277319605
RMSE: train - 1035.2864109711857 | test - 2146.820390887039
R2: train - 0.5809305588081777 | test - 0.3545016751084067


In [390]:
model = my_LinearRegression(random_state=21, solver='mbgd', batch_size=256)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f"MAE: train - {mean_absolute_error(y_train, y_pred_train)} | test - {mean_absolute_error(y_test, y_pred_test)}")
print(f"RMSE: train - {root_mean_squared_error(y_train, y_pred_train)} | test - {root_mean_squared_error(y_test, y_pred_test)}")
print(f"R2: train - {my_R2(y_train, y_pred_train)} | test - {my_R2(y_test, y_pred_test)}")

MAE: train - 711.0442256213722 | test - 816.3763583889406
RMSE: train - 1034.8436550832505 | test - 2148.2087530474987
R2: train - 0.581288924927764 | test - 0.3536665095425403


In [391]:
model = my_LinearRegression(random_state=21, solver='mbgd', batch_size=1)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f"MAE: train - {mean_absolute_error(y_train, y_pred_train)} | test - {mean_absolute_error(y_test, y_pred_test)}")
print(f"RMSE: train - {root_mean_squared_error(y_train, y_pred_train)} | test - {root_mean_squared_error(y_test, y_pred_test)}")
print(f"R2: train - {my_R2(y_train, y_pred_train)} | test - {my_R2(y_test, y_pred_test)}")

MAE: train - 718.7549082670777 | test - 823.6300957149004
RMSE: train - 1050.5312780194004 | test - 2154.3875136265997
R2: train - 0.5684978731512487 | test - 0.3499431441362878


In [392]:
model = my_LinearRegression(random_state=21, solver='sgd')
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f"MAE: train - {mean_absolute_error(y_train, y_pred_train)} | test - {mean_absolute_error(y_test, y_pred_test)}")
print(f"RMSE: train - {root_mean_squared_error(y_train, y_pred_train)} | test - {root_mean_squared_error(y_test, y_pred_test)}")
print(f"R2: train - {my_R2(y_train, y_pred_train)} | test - {my_R2(y_test, y_pred_test)}")

MAE: train - 718.7549082670777 | test - 823.6300957149004
RMSE: train - 1050.5312780194004 | test - 2154.3875136265997
R2: train - 0.5684978731512487 | test - 0.3499431441362878
